# Phase 1 — Notebook p1_02: Kernel & Throughput Bench

**What this notebook does — and does not do.** It does **not** train a real
model. It builds the actual shared building blocks every architecture (A,
B, C, D, S) reuses — RMSNorm, RoPE, grouped-query attention, SwiGLU, a KV
cache — at Model A's real size, and answers five infrastructure questions
on this specific GPU *before* we commit 6-12 real GPU-hours to training:

1. Which SDPA attention backend does PyTorch actually pick on a T4 (no
   FlashAttention-2 on this GPU's sm75 architecture), and how fast is it?
2. Does FlexAttention (needed for model B's sliding-window mask) even
   compile here?
3. Is our grouped-query attention implementation *correct* — does it match
   a slow-but-obviously-right reference?
4. Is the KV cache *correct* — does step-by-step generation match one full
   forward pass exactly?
5. What batch size and sequence length actually fit in a T4's 16GB, and
   how many tokens/sec does that get us?

Getting a wrong answer to any of these *during* a real 6-hour training run
is a wasted afternoon. Getting it here costs about 20 minutes.

**Runs on:** Kaggle GPU session (T4 x2). This is the first notebook in the
project that actually needs the accelerator.


## 1. Imports & GPU check

In [ ]:
import inspect
import json
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  "
          f"({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    print("\nWARNING: no GPU detected. Switch this notebook's accelerator "
          "to GPU T4 x2 in Settings before running the rest of this notebook.")


## 2. Config — Model A's real shape

Not a placeholder size — this is the actual architecture spec Model A will
train with, worked out to land at ~163M parameters (inside the 100-300M
range, with A deliberately at the upper end as the flagship). We benchmark
*this* shape, not a toy one, because a kernel that's fast at a small size
can behave completely differently at the real one (memory-bound vs
compute-bound regimes flip depending on shape).


In [ ]:
MODEL_A_CONFIG = dict(
    vocab_size=32_000,   # matches T1 (BPE), see p1_01_tokenizer.ipynb
    d_model=896,
    n_layer=16,
    n_head=14,
    n_kv_head=2,          # 7:1 GQA ratio — aggressive KV-cache savings
    d_head=64,            # 14 * 64 = 896 = d_model, as required
    d_ff=2432,            # SwiGLU hidden size, ~8/3 * d_model
    max_seq_len=2048,
)

OUTPUT_DIR = Path("/kaggle/working/phase1_bench")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert MODEL_A_CONFIG["n_head"] * MODEL_A_CONFIG["d_head"] == MODEL_A_CONFIG["d_model"], \
    "n_head * d_head must equal d_model"
assert MODEL_A_CONFIG["n_head"] % MODEL_A_CONFIG["n_kv_head"] == 0, \
    "n_head must divide evenly by n_kv_head for GQA grouping"

print("Model A config:", json.dumps(MODEL_A_CONFIG, indent=2))


## 3. The shared building blocks

These are not bench-only stand-ins — this is meant to become the real
`legallm/models/blocks.py`. Every design choice here matches the plan doc's
"design contracts": named `q_proj`/`k_proj`/`v_proj`/`o_proj`/`gate_proj`/
`up_proj`/`down_proj` layers so a PEFT method can target them by name later,
no biases, RMSNorm, tied embeddings.


In [ ]:
class RMSNorm(nn.Module):
    """RMSNorm instead of LayerNorm: one learned scale, no learned bias,
    no mean-subtraction — cheaper, and the modern-architecture standard
    this whole design follows (Llama, Mistral, etc.).
    """
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # Do the sum-of-squares reduction in fp32 even under autocast — a
        # T4 has no native bf16, so fp16 is what we'll actually train in,
        # and fp16 is more prone to overflowing on this particular sum than
        # on most other operations in the model.
        dtype = x.dtype
        x32 = x.float()
        norm = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0, device="cpu"):
    """Precompute RoPE's cos/sin tables once, up to the model's max
    context length, instead of recomputing them on every forward pass.
    """
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)  # [seq_len, d_head/2]
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """Rotate consecutive pairs of dimensions by a position-dependent
    angle. x: [B, n_head, T, d_head]; cos/sin: [T, d_head/2].
    """
    x1, x2 = x[..., 0::2], x[..., 1::2]
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated.flatten(-2)


# Whether this PyTorch build's SDPA can repeat KV heads internally
# (torch >= 2.5's `enable_gqa` flag). If it can, we let the kernel do the
# repeat without ever materializing it in memory — the entire point of
# GQA. If it can't, GQAAttention below transparently falls back to an
# explicit repeat_interleave, which works everywhere but costs the memory
# saving GQA is supposed to buy us. We find out which world we're in here,
# once, instead of guessing.
#
# `F.scaled_dot_product_attention` is a C-implemented builtin, so
# `inspect.signature` can't introspect it on every PyTorch build (it raises
# ValueError: "no signature found for builtin ..."). A tiny real call is
# the only detection that's actually reliable here.
try:
    _probe = torch.randn(1, 1, 2, 4)
    F.scaled_dot_product_attention(_probe, _probe, _probe, enable_gqa=True)
    _SDPA_HAS_ENABLE_GQA = True
except TypeError:
    _SDPA_HAS_ENABLE_GQA = False
print(f"This PyTorch build supports SDPA's native enable_gqa: {_SDPA_HAS_ENABLE_GQA}")


class GQAAttention(nn.Module):
    """Grouped-query attention: fewer KV heads than query heads, which
    shrinks the KV cache proportionally (here, 7x) at a small quality cost
    versus full multi-head attention.
    """
    def __init__(self, d_model, n_head, n_kv_head, d_head):
        super().__init__()
        self.n_head, self.n_kv_head, self.d_head = n_head, n_kv_head, d_head
        self.q_proj = nn.Linear(d_model, n_head * d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.o_proj = nn.Linear(n_head * d_head, d_model, bias=False)

    def forward(self, x, cos, sin, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if cache is not None:
            k, v = cache.update(k, v)

        # Causal masking only matters when we're attending over more than
        # one new query position (prefill). During single-token decoding
        # with a cache, every cached key is already a valid, earlier
        # position, so there's nothing left to mask.
        is_causal = q.shape[2] > 1

        if _SDPA_HAS_ENABLE_GQA:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal, enable_gqa=True)
        else:
            n_rep = self.n_head // self.n_kv_head
            k_rep = k.repeat_interleave(n_rep, dim=1)
            v_rep = v.repeat_interleave(n_rep, dim=1)
            out = F.scaled_dot_product_attention(q, k_rep, v_rep, is_causal=is_causal)

        out = out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head)
        return self.o_proj(out)


class SwiGLU(nn.Module):
    """Gated MLP: SiLU(gate) * up, then projected back down. Reuses more
    parameters per FLOP than a plain ReLU MLP of the same hidden size,
    which is why d_ff is set to ~8/3 x d_model rather than the classic 4x —
    that ratio keeps SwiGLU's parameter count roughly matched to a plain
    4x-ReLU MLP instead of adding params for free.
    """
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class Block(nn.Module):
    """One transformer layer: pre-norm attention, pre-norm MLP, both as
    residual additions."""
    def __init__(self, d_model, n_head, n_kv_head, d_head, d_ff):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_head, n_kv_head, d_head)
        self.mlp_norm = RMSNorm(d_model)
        self.mlp = SwiGLU(d_model, d_ff)

    def forward(self, x, cos, sin, cache=None):
        x = x + self.attn(self.attn_norm(x), cos, sin, cache)
        x = x + self.mlp(self.mlp_norm(x))
        return x


class KVCache:
    """A pre-allocated, fixed-size cache for one layer's keys and values.
    `update` writes the new slice in and returns everything filled so far —
    generation never has to recompute earlier positions.
    """
    def __init__(self, batch_size, n_kv_head, max_seq_len, d_head, device, dtype):
        shape = (batch_size, n_kv_head, max_seq_len, d_head)
        self.k = torch.zeros(shape, device=device, dtype=dtype)
        self.v = torch.zeros(shape, device=device, dtype=dtype)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


print("Building blocks defined: RMSNorm, RoPE, GQAAttention, SwiGLU, Block, KVCache.")


## 4. The model itself

Assembles the blocks above into Model A's real shape. Weight tying (input
embedding = output head) and the residual-scaling init both matter for
training stability at this depth, so we set them up correctly here even
though this notebook never actually trains on real data.


In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([
            Block(config["d_model"], config["n_head"], config["n_kv_head"],
                  config["d_head"], config["d_ff"])
            for _ in range(config["n_layer"])
        ])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight  # weight tying: one set of params, not two

        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        # Residual-scaling init: output-facing projections get a smaller
        # initial scale so the residual stream doesn't blow up as depth
        # grows. Standard trick (GPT-2, Llama), matters more the deeper
        # the model gets — and 16 layers is deep enough for it to matter.
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def new_kv_caches(self, batch_size, device, dtype):
        """One KVCache per layer, all pre-allocated up front — this is
        what a real generation loop calls once before decoding starts.
        """
        return [
            KVCache(batch_size, self.config["n_kv_head"], self.config["max_seq_len"],
                    self.config["d_head"], device, dtype)
            for _ in range(self.config["n_layer"])
        ]

    def forward(self, ids, caches=None):
        B, T = ids.shape
        x = self.embed(ids)
        pos_offset = caches[0].pos if caches is not None else 0
        cos = self.rope_cos[pos_offset:pos_offset + T].to(x.device)
        sin = self.rope_sin[pos_offset:pos_offset + T].to(x.device)
        for i, blk in enumerate(self.blocks):
            cache = caches[i] if caches is not None else None
            x = blk(x, cos, sin, cache)
        x = self.norm_f(x)
        return self.head(x)


model = MiniGPT(MODEL_A_CONFIG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
n_params_notied = n_params + model.embed.weight.numel()  # what it'd be WITHOUT tying, for comparison
print(f"Model A parameter count (tied embeddings): {n_params / 1e6:.1f}M")
print(f"(without weight tying it would be: {n_params_notied / 1e6:.1f}M)")
assert 50e6 <= n_params <= 300e6, "outside the competition's allowed parameter range!"


## 5. SDPA backend probe

PyTorch's `scaled_dot_product_attention` picks a backend kernel
automatically, but "automatically" isn't the same as "the one you'd
expect" — a T4 (sm75) has no FlashAttention-2 support at all, so the
question is really: does it fall back to the memory-efficient kernel
(good) or to the slow, naive math kernel (bad, and worth knowing about
now rather than discovering it from an oddly slow training run later)?


In [ ]:
try:
    from torch.nn.attention import SDPBackend, sdpa_kernel
    _NEW_SDPA_API = True
except ImportError:
    _NEW_SDPA_API = False

BACKENDS_TO_TRY = ["flash", "mem_efficient", "math"]


def time_sdpa_backend(backend_name, q, k, v, n_iters=20):
    """Time `n_iters` calls to SDPA forced onto one specific backend.
    Returns None if that backend isn't usable for this input on this GPU
    (e.g. flash attention simply refusing to run on sm75) rather than
    crashing the whole probe.
    """
    try:
        if _NEW_SDPA_API:
            backend_enum = {"flash": SDPBackend.FLASH_ATTENTION,
                             "mem_efficient": SDPBackend.EFFICIENT_ATTENTION,
                             "math": SDPBackend.MATH}[backend_name]
            ctx = sdpa_kernel(backend_enum)
        else:
            flags = {"flash": dict(enable_flash=True, enable_math=False, enable_mem_efficient=False),
                      "mem_efficient": dict(enable_flash=False, enable_math=False, enable_mem_efficient=True),
                      "math": dict(enable_flash=False, enable_math=True, enable_mem_efficient=False)}[backend_name]
            ctx = torch.backends.cuda.sdp_kernel(**flags)

        with ctx:
            torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(n_iters):
                F.scaled_dot_product_attention(q, k, v, is_causal=True)
            torch.cuda.synchronize()
            return (time.time() - t0) / n_iters
    except RuntimeError as exc:
        print(f"  {backend_name}: not usable here ({exc})")
        return None


B, n_head, T, d_head = 8, MODEL_A_CONFIG["n_head"], 1024, MODEL_A_CONFIG["d_head"]
q = torch.randn(B, n_head, T, d_head, device=DEVICE, dtype=torch.float16)
k = torch.randn(B, n_head, T, d_head, device=DEVICE, dtype=torch.float16)
v = torch.randn(B, n_head, T, d_head, device=DEVICE, dtype=torch.float16)

backend_results = {}
print(f"Timing each SDPA backend at B={B}, T={T} (fp16)...")
for name in BACKENDS_TO_TRY:
    t = time_sdpa_backend(name, q, k, v)
    backend_results[name] = t
    if t is not None:
        print(f"  {name:14s}: {t * 1000:.2f} ms/call")

fastest = min((n for n, t in backend_results.items() if t is not None), key=lambda n: backend_results[n])
print(f"\nFastest available backend on this GPU: {fastest}")
print("This matches expectations if it's 'mem_efficient' — T4 (sm75) has no FlashAttention-2 support.")


## 6. FlexAttention compile check

Model B needs a sliding-window attention pattern. FlexAttention is the
clean way to express that, but it's a newer, `torch.compile`-based API —
worth confirming it actually compiles and runs correctly on this GPU/torch
combination now, so B's notebook isn't the first place we find out it
doesn't.


In [ ]:
FLEX_ATTENTION_WORKS = False

try:
    from torch.nn.attention.flex_attention import flex_attention, create_block_mask

    def sliding_window_mask(window=512):
        def mask_mod(b, h, q_idx, kv_idx):
            causal = kv_idx <= q_idx
            within_window = q_idx - kv_idx < window
            return causal & within_window
        return mask_mod

    block_mask = create_block_mask(sliding_window_mask(512), B=None, H=None, Q_LEN=T, KV_LEN=T, device=DEVICE)
    compiled_flex = torch.compile(flex_attention)

    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(10):
        out = compiled_flex(q, k, v, block_mask=block_mask)
    torch.cuda.synchronize()
    flex_time = (time.time() - t0) / 10

    print(f"FlexAttention (sliding window) compiled and ran: {flex_time * 1000:.2f} ms/call")
    FLEX_ATTENTION_WORKS = True

except Exception as exc:
    print(f"FlexAttention did not work on this GPU/torch combination: {exc}")
    print("Fallback plan for model B: a manual boolean sliding-window mask "
          "passed to plain SDPA (slower, but always available). Noting "
          "this in the plan doc.")

print(f"\nFLEX_ATTENTION_WORKS = {FLEX_ATTENTION_WORKS}")


## 7. GQA correctness check

Compares our fast `GQAAttention` (which either uses SDPA's native
`enable_gqa` or falls back to `repeat_interleave`, per Section 3) against
an intentionally slow, obviously-correct reference that always does the
repeat explicitly. Sharing the exact same weights between the two means
any difference in output is a real bug, not a training-randomness
artifact.


In [ ]:
class GQAAttentionReference(nn.Module):
    """Deliberately naive: always repeats KV heads explicitly, never
    relies on enable_gqa. Slower and more memory-hungry on purpose — this
    exists only to check correctness, never used in real training.
    """
    def __init__(self, gqa_module):
        super().__init__()
        self.m = gqa_module  # reuse the fast module's own weights

    def forward(self, x, cos, sin):
        m = self.m
        B, T, _ = x.shape
        q = m.q_proj(x).view(B, T, m.n_head, m.d_head).transpose(1, 2)
        k = m.k_proj(x).view(B, T, m.n_kv_head, m.d_head).transpose(1, 2)
        v = m.v_proj(x).view(B, T, m.n_kv_head, m.d_head).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        n_rep = m.n_head // m.n_kv_head
        k = k.repeat_interleave(n_rep, dim=1)
        v = v.repeat_interleave(n_rep, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).reshape(B, T, m.n_head * m.d_head)
        return m.o_proj(out)


test_attn = GQAAttention(MODEL_A_CONFIG["d_model"], MODEL_A_CONFIG["n_head"],
                          MODEL_A_CONFIG["n_kv_head"], MODEL_A_CONFIG["d_head"]).to(DEVICE)
reference_attn = GQAAttentionReference(test_attn).to(DEVICE)

test_cos, test_sin = build_rope_cache(64, MODEL_A_CONFIG["d_head"], device=DEVICE)
test_x = torch.randn(2, 64, MODEL_A_CONFIG["d_model"], device=DEVICE)

with torch.no_grad():
    fast_out = test_attn(test_x, test_cos, test_sin)
    ref_out = reference_attn(test_x, test_cos, test_sin)

max_diff = (fast_out - ref_out).abs().max().item()
print(f"Max absolute difference between fast GQA and reference: {max_diff:.2e}")
assert torch.allclose(fast_out, ref_out, atol=1e-4, rtol=1e-4), \
    "GQA fast path does not match the reference — real bug, fix before training!"
print("GQA correctness check passed.")


## 8. KV-cache correctness check

The classic silent bug in a from-scratch transformer: generation looks
plausible, loss curves look fine, and the model is still subtly wrong
because the cache is misaligned. The only real test is comparing
step-by-step generation against one full forward pass over the same
tokens — they must produce numerically identical logits.


In [ ]:
model.eval()
test_ids = torch.randint(0, MODEL_A_CONFIG["vocab_size"], (1, 32), device=DEVICE)

with torch.no_grad():
    full_logits = model(test_ids)  # one pass over all 32 positions, no cache

    caches = model.new_kv_caches(batch_size=1, device=DEVICE, dtype=next(model.parameters()).dtype)
    stepped_logits = []
    for t in range(test_ids.shape[1]):
        step_logits = model(test_ids[:, t:t + 1], caches=caches)  # one new token at a time
        stepped_logits.append(step_logits)
    stepped_logits = torch.cat(stepped_logits, dim=1)

max_diff = (full_logits - stepped_logits).abs().max().item()
print(f"Max absolute difference between full-forward and step-by-step-with-cache logits: {max_diff:.2e}")
assert torch.allclose(full_logits, stepped_logits, atol=1e-3, rtol=1e-3), \
    "KV cache does not match a full forward pass — real bug, fix before training!"
print("KV-cache correctness check passed.")
model.train()


## 9. fp16 stability check

A T4 has no native bf16, so real training runs in fp16 with a
`GradScaler`. A handful of dummy forward/backward steps here, watching for
NaN or Inf in the loss, catches a badly-scaled operation before it costs
hours of wasted training.


In [ ]:
scaler = torch.cuda.amp.GradScaler()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

print("Running 5 dummy fp16 training steps...")
for step in range(5):
    batch = torch.randint(0, MODEL_A_CONFIG["vocab_size"], (4, 512), device=DEVICE)
    targets = torch.randint(0, MODEL_A_CONFIG["vocab_size"], (4, 512), device=DEVICE)

    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = model(batch)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    is_finite = torch.isfinite(loss).item()
    print(f"  step {step}: loss={loss.item():.4f}  finite={is_finite}")
    assert is_finite, "Got a NaN/Inf loss under fp16 — revisit norm placement or init before real training."

print("\nfp16 stability check passed (on random data — real data may still need a lower peak LR; "
      "this only rules out a structural instability in the architecture itself).")


## 10. Batch size / sequence length throughput sweep

The number that actually matters for tomorrow's real training run: how
many tokens/sec do we get, and how much memory does it cost, across a
grid of batch sizes and context lengths? We leave real headroom below the
GPU's 16GB — this benchmark loop doesn't include an optimizer step's extra
memory (Adam keeps two extra fp32 copies of every parameter), so a config
that looks like it barely fits here would not fit once real training adds
that overhead.


In [ ]:
def measure_throughput(batch_size, seq_len, n_iters=5):
    """Returns (tokens_per_sec, peak_memory_gb) for one forward+backward
    configuration, or (None, None) if it doesn't fit in memory at all.
    """
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        batch = torch.randint(0, MODEL_A_CONFIG["vocab_size"], (batch_size, seq_len), device=DEVICE)
        targets = torch.randint(0, MODEL_A_CONFIG["vocab_size"], (batch_size, seq_len), device=DEVICE)

        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_iters):
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(batch)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        torch.cuda.synchronize()
        elapsed = time.time() - t0

        tokens_per_sec = (batch_size * seq_len * n_iters) / elapsed
        peak_gb = torch.cuda.max_memory_allocated() / 1e9
        return tokens_per_sec, peak_gb
    except torch.cuda.OutOfMemoryError:
        return None, None


BATCH_SIZES = [2, 4, 8, 16]
SEQ_LENS = [512, 1024, 2048]
MEMORY_BUDGET_GB = 13.0  # leaves ~3GB headroom on a 16GB T4 for Adam's extra fp32 state during real training

sweep_results = []
for seq_len in SEQ_LENS:
    for batch_size in BATCH_SIZES:
        tps, peak_gb = measure_throughput(batch_size, seq_len)
        status = "OOM" if tps is None else ("fits" if peak_gb <= MEMORY_BUDGET_GB else "over budget")
        print(f"  batch={batch_size:3d}  seq_len={seq_len:5d}  "
              f"-> {'OOM' if tps is None else f'{tps:9,.0f} tok/s, {peak_gb:.2f} GB'}  [{status}]")
        sweep_results.append(dict(batch_size=batch_size, seq_len=seq_len,
                                   tokens_per_sec=tps, peak_memory_gb=peak_gb, status=status))


## 11. Plot + pick the recommendation

The config we actually want is the highest-throughput one that still fits
inside the memory budget — not just the highest-throughput one overall,
which might be a config that would OOM the moment real training's
optimizer state gets added on top.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for seq_len in SEQ_LENS:
    xs = [r["batch_size"] for r in sweep_results if r["seq_len"] == seq_len and r["tokens_per_sec"]]
    ys = [r["tokens_per_sec"] for r in sweep_results if r["seq_len"] == seq_len and r["tokens_per_sec"]]
    ax.plot(xs, ys, marker="o", label=f"seq_len={seq_len}")
ax.set_xlabel("Batch size (per GPU)")
ax.set_ylabel("Tokens / sec")
ax.set_title("Model A throughput on one T4 (fp16, fwd+bwd+optimizer step)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plot_path = OUTPUT_DIR / "throughput_sweep.png"
fig.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved plot -> {plot_path}")

fitting = [r for r in sweep_results if r["status"] == "fits"]
best = max(fitting, key=lambda r: r["tokens_per_sec"]) if fitting else None

if best:
    print(f"\nRecommended config: batch_size={best['batch_size']}, seq_len={best['seq_len']}"
          f"  ({best['tokens_per_sec']:,.0f} tok/s, {best['peak_memory_gb']:.2f} GB peak, "
          f"under the {MEMORY_BUDGET_GB} GB budget)")
    print("With 2x T4 via DDP, expect roughly 2x this throughput (data-parallel, not compute-shared).")
else:
    print("\nNothing in the sweep stayed under budget — lower MEMORY_BUDGET_GB's assumptions "
          "or shrink the batch/seq_len grid and re-run.")


## 12. Save the bench report

Written once, read by `p1_03_model_A.ipynb` (and every later architecture
notebook) instead of re-running this whole sweep each time.


In [ ]:
report = dict(
    model_config=MODEL_A_CONFIG,
    n_params=int(n_params),
    fastest_sdpa_backend=fastest,
    sdpa_backend_timings_ms={k: (v * 1000 if v is not None else None) for k, v in backend_results.items()},
    flex_attention_works=FLEX_ATTENTION_WORKS,
    gqa_correctness_max_diff=max_diff,
    recommended_batch_size=best["batch_size"] if best else None,
    recommended_seq_len=best["seq_len"] if best else None,
    recommended_tokens_per_sec=best["tokens_per_sec"] if best else None,
    memory_budget_gb=MEMORY_BUDGET_GB,
    full_sweep=sweep_results,
)

report_path = OUTPUT_DIR / "bench_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Saved -> {report_path}")
print(json.dumps({k: v for k, v in report.items() if k != "full_sweep"}, indent=2))


## 13. Publish as a Kaggle Dataset

Same push pattern as the previous two notebooks.


In [ ]:
import subprocess
from kaggle_secrets import UserSecretsClient

KAGGLE_USERNAME = "jatiniitk"
DATASET_SLUG = "legallm-bench"

try:
    kaggle_token = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
except Exception as exc:
    kaggle_token = None
    print(f"Could not read the 'KAGGLE_API_TOKEN' secret ({exc}). "
          f"Add it under Add-ons -> Secrets (and check the box on!) before running this cell.")

if kaggle_token:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    token_path = kaggle_dir / "access_token"
    token_path.write_text(kaggle_token)
    token_path.chmod(0o600)

    dataset_id = f"{KAGGLE_USERNAME}/{DATASET_SLUG}"
    metadata = {"title": DATASET_SLUG, "id": dataset_id, "licenses": [{"name": "CC0-1.0"}]}
    with open(OUTPUT_DIR / "dataset-metadata.json", "w") as f:
        json.dump(metadata, f)

    exists = subprocess.run(
        ["kaggle", "datasets", "status", dataset_id], capture_output=True, text=True,
    ).returncode == 0

    if exists:
        print(f"Dataset {dataset_id} already exists -> pushing a new version.")
        cmd = ["kaggle", "datasets", "version", "-p", str(OUTPUT_DIR),
               "-m", f"Bench run at {time.strftime('%Y-%m-%d %H:%M:%S')}", "-r", "zip"]
    else:
        print(f"Dataset {dataset_id} does not exist yet -> creating it.")
        cmd = ["kaggle", "datasets", "create", "-p", str(OUTPUT_DIR), "-r", "zip"]

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    else:
        print(f"Done. Attach it to future notebooks via 'Add Data' -> search '{DATASET_SLUG}'.")
else:
    print("Skipped: missing KAGGLE_API_TOKEN secret.")


## 14. Summary

| Check | Result |
| --- | --- |
| Model A parameter count | printed in Section 4 — must be 50-300M |
| Fastest SDPA backend on this T4 | printed in Section 5 |
| FlexAttention (needed by model B) | printed in Section 6 |
| GQA correctness | passed if Section 7 didn't raise |
| KV-cache correctness | passed if Section 8 didn't raise |
| fp16 stability | passed if Section 9 didn't raise |
| Recommended batch_size / seq_len | printed in Section 11, saved in `bench_report.json` |

**If FlexAttention failed:** model B's notebook uses a manual boolean
sliding-window mask with plain SDPA instead — slower, but correct and
always available. Note this in the plan doc either way.

**Next notebook:** `p1_03_model_A.ipynb` — the real training run, using
these exact blocks and the batch size this notebook recommended.
